# One-day pilot — Bitcoin decentralisation

The whole project in miniature: pull 1,000 random blocks, attribute each to a
pool by its coinbase tag, compute the Nakamoto coefficient, and plot it.

**This is meant to be rough.** The point isn't a publishable result — it's to
hit every obstacle once (messy tags, unattributable blocks, the unknown-handling
choice) on a tiny scale, so the real pipeline holds no surprises. Run it top to
bottom, then write down what you notice in the last cell.

Prereqs (already done): `pip install -r requirements.txt`, and
`gcloud auth application-default login`.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

# Your project ID — the one `gcloud config get-value project` printed earlier.
# Querying public data is free, but it must be billed to YOUR project.
PROJECT_ID = "your-project-id"   # <-- EDIT THIS

client = bigquery.Client(project=PROJECT_ID)
print("BigQuery client ready, billing project:", client.project)

## Step 1 — pull 1,000 random blocks

We only need the coinbase tag, which lives in `coinbase_param` on the **`blocks`**
table (~950k rows). We never touch the giant `transactions` table here, so the
scan is tiny. We check the cost with a *dry run* before spending anything — make
this a reflex for every query you write.

In [ ]:
QUERY = """
SELECT
  number    AS height,
  timestamp,
  coinbase_param
FROM `bigquery-public-data.crypto_bitcoin.blocks`
WHERE coinbase_param IS NOT NULL
ORDER BY RAND()
LIMIT 1000
"""

# DRY RUN: estimate bytes scanned without running the query
dry = client.query(QUERY, bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
gb = dry.total_bytes_processed / 1e9
print(f"This query will scan ~{gb:.4f} GB ({gb*1000:.1f} MB).  Free tier = 1000 GB/month.")
assert gb < 5, "Unexpectedly large scan — investigate before running for real."

In [ ]:
# Looks cheap — run it for real
blocks_df = client.query(QUERY).to_dataframe()
print("shape:", blocks_df.shape)
blocks_df.head()

## Step 2 — decode the coinbase tag

`coinbase_param` is a hex string. The pool tag is human-readable text buried
among non-printable bytes (block height, extranonce). We decode to best-effort
ASCII, turning unreadable bytes into `.` so the tag stands out. **Eyeball the
output** — this is where you learn what real coinbase tags actually look like.

In [ ]:
def coinbase_to_ascii(hex_str):
    """Hex coinbase_param -> best-effort printable ASCII (non-printable -> '.')."""
    if not hex_str:
        return ""
    try:
        raw = bytes.fromhex(hex_str)
    except (ValueError, TypeError):
        return str(hex_str)
    return "".join(chr(b) if 32 <= b < 127 else "." for b in raw)

blocks_df["coinbase_ascii"] = blocks_df["coinbase_param"].apply(coinbase_to_ascii)

# eyeball 25 of them
for s in blocks_df["coinbase_ascii"].head(25):
    print(s)

## Step 3 — match against 5 hand-typed pools

A deliberately crude tag matcher over five well-known pools. **This crudeness is
the point of your whole dissertation** — which reference list, which matching
rule, and how you treat everything that doesn't match (D2, and the heuristics
H1–H5) are exactly the choices you'll vary later. For now: substring match, five
pools, everything else `unknown`.

In [ ]:
POOL_TAGS = {
    "F2Pool":    ["f2pool"],
    "AntPool":   ["antpool"],
    "ViaBTC":    ["viabtc"],
    "SlushPool": ["slush"],     # also catches the older "/slush/" tag
    "BTC.com":   ["btc.com", "btccom"],
}

def attribute_by_tag(ascii_script):
    s = ascii_script.lower()
    for pool, needles in POOL_TAGS.items():
        if any(n in s for n in needles):
            return pool
    return "unknown"

blocks_df["pool"] = blocks_df["coinbase_ascii"].apply(attribute_by_tag)

counts = blocks_df["pool"].value_counts()
print(counts.to_string())
unknown_share = counts.get("unknown", 0) / len(blocks_df)
print(f"\nunknown: {unknown_share:.1%} of blocks")

## Step 4 — Nakamoto coefficient (and the choice that changes it)

The Nakamoto coefficient = the smallest number of entities jointly controlling
>50% of blocks. We compute it **two ways**: counting `unknown` as a single
entity, and excluding it. These are two of your D3 variants (U1 vs U2). Watch
whether the two numbers disagree — *that gap, on real data, is your entire
dissertation in one line.*

In [ ]:
def nakamoto_coefficient(entity_counts):
    shares = pd.Series(entity_counts).sort_values(ascending=False)
    total = shares.sum()
    if total == 0:
        return 0
    cum = 0
    for i, (_e, n) in enumerate(shares.items(), start=1):
        cum += n
        if cum / total > 0.5:
            return i
    return len(shares)

nak_incl  = nakamoto_coefficient(counts)
nak_known = nakamoto_coefficient(counts.drop("unknown", errors="ignore"))

print(f"Nakamoto, unknown as one entity (U1): {nak_incl}")
print(f"Nakamoto, unknown excluded      (U2): {nak_known}")

## Step 5 — plot it

Sorted block share per entity (bars) with the running cumulative share (line).
Where the cumulative line crosses the dashed 50% mark *is* the Nakamoto
coefficient — you can read it straight off the x-axis.

In [ ]:
import os

shares = (counts / counts.sum()).sort_values(ascending=False)
cum = shares.cumsum()

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#999999" if e == "unknown" else "#d1495b" for e in shares.index]
ax.bar(range(len(shares)), shares.values, color=colors, label="share per entity")
ax.plot(range(len(shares)), cum.values, "o-", color="#1f4e5f", label="cumulative share")
ax.axhline(0.5, ls="--", color="black", lw=1)
ax.text(len(shares) - 1, 0.52, "50% threshold", ha="right", fontsize=9)
ax.set_xticks(range(len(shares)))
ax.set_xticklabels(shares.index, rotation=30, ha="right")
ax.set_ylabel("share of blocks")
ax.set_title(f"Pilot: pool concentration (n={len(blocks_df)})  |  "
             f"Nakamoto: known-only={nak_known}, incl-unknown={nak_incl}")
ax.legend()
fig.tight_layout()

os.makedirs("../figures", exist_ok=True)   # assumes you run this from notebooks/
fig.savefig("../figures/pilot_shares.png", dpi=120, bbox_inches="tight")
plt.show()

## What to write down (do this now, while it's fresh)

Per your roadmap: 5–10 observations about quirks you noticed. Prompts:

- What fraction came back `unknown`? Higher or lower than you expected?
- Scroll the decoded tags in Step 2 — do any look like a real pool you *didn't*
  put in the list of five? (That's the reference-list completeness problem, live.)
- Did any tags look ambiguous, malformed, or share text with another pool?
- Did the two Nakamoto numbers in Step 4 differ? By how much?
- Anything about the height/extranonce noise, encoding, or empty tags?

**Why each of these is the thesis, not a side-quirk:**
- the unknown rate → dimension D3 and the U1/U2/U3 treatments
- the messy/ambiguous tags → dimension D2 and heuristics H1–H5
- the gap between the two Nakamoto numbers → the sensitivity question itself (RQ1/RQ2)

_Your observations:_

1.
2.
3.
